<!-- TEACHABLE_PROJECT_NOTE -->
# Bronze Notebook - Raw Data Entry Layer


# Bronze Layer: Raw Data Inspection

This notebook opens the original `data.csv` file without changing it. The goal of the bronze layer is to understand the raw schema, preview records, and identify cleaning work needed before creating the silver dataset.


## Setup

Import pandas and define the raw data path in one place so later cells are easy to adjust.


In [2]:
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path("data.csv")


## Load Raw Data

Read the CSV exactly as provided. The preview confirms that the file loaded correctly and shows the first few rows.


In [3]:
df_bronze = pd.read_csv(RAW_DATA_PATH)
df_bronze.head()


,post_id,platform,timestamp,crisis_phase,topic,sentiment_label,sentiment_score,emotion,stance_label,claim_type,...,impressions_estimate,reach_estimate,engagement_velocity,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score,region,language
0,1,Twitter,2025-06-05 18:23:00,Escalation,Service disruption,Neutral,-0.108,Fear,Support,Opinion,...,30897,10254,7.808,0.308,0.712,0.503,0.749,0.036,Global,English
1,2,Reddit,2025-08-11 21:56:00,Recovery,User complaints,Neutral,0.063,Trust,Neutral,Fact Claim,...,34685,17812,20.347,0.110,0.721,0.396,0.585,0.240,Global,English
2,3,Twitter,2025-07-05 02:09:00,Peak,User complaints,Negative,-0.364,Frustration,Question,Fact Claim,...,109789,45897,143.045,0.746,0.105,0.676,0.372,0.452,Global,English
3,4,Reddit,2025-08-27 18:01:00,Response,Privacy concerns,Neutral,0.176,Confusion,Criticize,Observation,...,39077,14778,31.070,0.095,0.737,0.239,0.281,0.022,Global,English
4,5,Reddit,2025-07-10 10:20:00,Recovery,Service disruption,Positive,0.516,Trust,Criticize,Fact Claim,...,71736,52714,21.411,0.147,0.859,0.356,0.479,0.082,Global,English


## Schema Overview

`info()` shows column names, row counts, missing values, and inferred data types. Use this to decide which columns need type conversion or missing-value handling in the silver layer.


In [4]:
df_bronze.head(20)

,post_id,platform,timestamp,crisis_phase,topic,sentiment_label,sentiment_score,emotion,stance_label,claim_type,...,impressions_estimate,reach_estimate,engagement_velocity,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score,region,language
0,1,Twitter,2025-06-05 18:23:00,Escalation,Service disruption,Neutral,-0.108,Fear,Support,Opinion,...,30897,10254,7.808,0.308,0.712,0.503,0.749,0.036,Global,English
1,2,Reddit,2025-08-11 21:56:00,Recovery,User complaints,Neutral,0.063,Trust,Neutral,Fact Claim,...,34685,17812,20.347,0.110,0.721,0.396,0.585,0.240,Global,English
2,3,Twitter,2025-07-05 02:09:00,Peak,User complaints,Negative,-0.364,Frustration,Question,Fact Claim,...,109789,45897,143.045,0.746,0.105,0.676,0.372,0.452,Global,English
3,4,Reddit,2025-08-27 18:01:00,Response,Privacy concerns,Neutral,0.176,Confusion,Criticize,Observation,...,39077,14778,31.070,0.095,0.737,0.239,0.281,0.022,Global,English
4,5,Reddit,2025-07-10 10:20:00,Recovery,Service disruption,Positive,0.516,Trust,Criticize,Fact Claim,...,71736,52714,21.411,0.147,0.859,0.356,0.479,0.082,Global,English
5,6,Facebook,2025-07-19 22:15:00,Escalation,Privacy concerns,Neutral,0.022,Fear,Criticize,Opinion,...,104153,29269,99.894,0.122,0.925,0.870,0.935,0.234,Global,English
6,7,Twitter,2025-07-18 11:25:00,Escalation,Official communication,Negative,-0.335,Frustration,Criticize,Rumor,...,28035,12590,14.194,0.508,0.636,0.998,0.868,0.165,Global,English
7,8,Reddit,2025-06-08 14:56:00,Escalation,Privacy concerns,Negative,-0.316,Frustration,Criticize,Observation,...,5975,4225,8.136,0.213,0.553,0.897,0.651,0.825,Global,English
8,9,Facebook,2025-07-03 19:50:00,Recovery,Official communication,Neutral,0.126,Trust,Criticize,Fact Claim,...,47217,33628,13.722,0.132,0.580,0.183,0.881,0.176,Global,English
9,10,Twitter,2025-06-05 16:43:00,Peak,User complaints,Negative,-0.776,Fear,Neutral,Observation,...,84162,50096,56.563,0.096,0.961,0.591,0.215,0.561,Global,English


In [5]:
df_bronze.columns


Index(['post_id', 'platform', 'timestamp', 'crisis_phase', 'topic',
       'sentiment_label', 'sentiment_score', 'emotion', 'stance_label',
       'claim_type', 'follower_count', 'following_count', 'account_age_days',
       'verified_flag', 'source_type', 'is_share', 'parent_post_id',
       'cascade_depth', 'likes', 'shares', 'comments', 'impressions_estimate',
       'reach_estimate', 'engagement_velocity', 'misinformation_probability',
       'credibility_score', 'uncertainty_index', 'subjectivity_score',
       'toxicity_score', 'region', 'language'],
      dtype='str')

## Numeric Profile

`describe()` summarizes numeric distributions such as counts, averages, ranges, and quartiles. These values help spot suspicious outliers before transformation.


In [6]:
df_bronze.describe()


,post_id,sentiment_score,follower_count,following_count,account_age_days,verified_flag,is_share,parent_post_id,cascade_depth,likes,shares,comments,impressions_estimate,reach_estimate,engagement_velocity,misinformation_probability,credibility_score,uncertainty_index,subjectivity_score,toxicity_score
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.0,50000.000000,10879.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,25000.500000,-0.165099,2153.679920,827.192820,1224.207080,0.0,0.217580,12562.891442,0.983780,1962.063420,782.049820,389.588880,70855.276660,35443.975920,90.253141,0.237569,0.727135,0.488091,0.574731,0.286666
std,14433.901067,0.387713,1615.104051,616.308535,903.328119,0.0,0.412604,10990.290817,2.150658,1541.000882,615.600373,308.320741,61062.076811,33468.119437,258.870828,0.198357,0.176647,0.272821,0.216317,0.218026
min,1.000000,-1.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,29.000000,12.000000,0.364000,0.000000,0.100000,0.050000,0.200000,0.000000
25%,12500.750000,-0.439000,867.000000,334.000000,500.000000,0.0,0.000000,3415.500000,0.000000,856.000000,343.000000,168.000000,28512.250000,12940.750000,18.477000,0.100000,0.629000,0.255000,0.388000,0.119000
50%,25000.500000,-0.145000,1828.000000,708.000000,1053.000000,0.0,0.000000,9422.000000,0.000000,1715.000000,681.000000,340.000000,55820.500000,26323.000000,32.035500,0.201000,0.745000,0.458000,0.575000,0.238000
75%,37500.250000,0.109000,3110.000000,1198.000000,1775.000000,0.0,0.000000,19191.500000,0.000000,2566.000000,1026.000000,512.000000,94380.250000,46705.000000,68.828500,0.300000,0.862000,0.722000,0.762000,0.372000
max,50000.000000,0.700000,10495.000000,4403.000000,6119.000000,0.0,1.000000,49285.000000,8.000000,7500.000000,3000.000000,1500.000000,411641.000000,295080.000000,9196.494000,0.950000,0.980000,1.000000,0.950000,0.850000


# Save Data

In [7]:
df_bronze.to_csv("Bronze.csv", index=False)